# XGBoost Frequency Modeling

## Insurance Pricing Project

**Objective**

Develop a machine learning model to predict claim frequency using XGBoost with a Poisson objective function. The model will serve as a challenger to the classical Poisson and Negative Binomial GLMs developed in previous stages of the project.

The ultimate goal is to improve prediction accuracy while preserving the ability to estimate insurance claim frequencies required for premium calculation.

# Business Context

Insurance companies estimate future claim frequency to determine expected losses and calculate fair premiums.

Classical Generalized Linear Models (GLMs) are widely used because they are interpretable and regulator-friendly. However, they assume relatively simple relationships between predictors and the response.

Tree-based gradient boosting models such as XGBoost can automatically capture:

- nonlinear relationships
- complex feature interactions
- threshold effects
- heterogeneous customer behavior

This notebook evaluates whether these additional modeling capabilities improve prediction performance compared to traditional actuarial models.

# Project Objectives

In this notebook we will:

- Prepare the modeling dataset
- Handle missing values
- Encode categorical variables
- Train an XGBoost Poisson model
- Predict claim frequency
- Evaluate model performance
- Compare results against Poisson GLM and Negative Binomial GLM
- Interpret feature importance

# 1. Import Required Libraries

In this section, we import all the libraries required for data manipulation, visualization, preprocessing, model development, and evaluation.

The libraries used include:

- **Pandas** and **NumPy** for data manipulation.
- **Matplotlib** and **Seaborn** for visualization.
- **Scikit-learn** for preprocessing, train-test splitting, and evaluation metrics.
- **XGBoost** for building the machine learning frequency model.

Importing all required packages at the beginning of the notebook improves readability and ensures reproducibility.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Evaluation Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance
)

# XGBoost
from xgboost import XGBRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# 2. Load the Processed Dataset

The processed dataset created during the data preparation phase is loaded into the notebook.

This dataset has already undergone:

- Data cleaning
- Missing value assessment
- Variable validation
- Exploratory data analysis

Using a processed dataset ensures that model development focuses on predictive performance rather than data quality issues.

In [2]:
df = pd.read_csv(r"D:\Insurance_Pricing_Project\Data\freMTPL2freq.csv")

# 3. Dataset Overview

Before model development, it is good practice to verify the structure of the dataset.

The following checks are performed:

- Number of observations
- Number of variables
- Data types
- Missing values
- Basic descriptive statistics

These checks help ensure that the dataset is suitable for machine learning.

In [3]:
print(f"Dataset Shape: {df.shape}")

display(df.info())

display(df.describe())

display(df.isnull().sum())

Dataset Shape: (678013, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 678013 entries, 0 to 678012
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   IDpol       678013 non-null  float64
 1   ClaimNb     678013 non-null  int64  
 2   Exposure    678013 non-null  float64
 3   Area        678013 non-null  object 
 4   VehPower    678013 non-null  int64  
 5   VehAge      678013 non-null  int64  
 6   DrivAge     678013 non-null  int64  
 7   BonusMalus  678013 non-null  int64  
 8   VehBrand    678013 non-null  object 
 9   VehGas      678013 non-null  object 
 10  Density     678013 non-null  int64  
 11  Region      678013 non-null  object 
dtypes: float64(2), int64(6), object(4)
memory usage: 62.1+ MB


None

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,Density
count,6.780130e+05,678013.000000,678013.000000,678013.000000,678013.000000,678013.000000,678013.000000,678013.000000
mean,2.621857e+06,0.053247,0.528750,6.454631,7.044265,45.499122,59.761502,1792.422405
std,1.641783e+06,0.240117,0.364442,2.050906,5.666232,14.137444,15.636658,3958.646564
min,1.000000e+00,0.000000,0.002732,4.000000,0.000000,18.000000,50.000000,1.000000
25%,1.157951e+06,0.000000,0.180000,5.000000,2.000000,34.000000,50.000000,92.000000
50%,2.272152e+06,0.000000,0.490000,6.000000,6.000000,44.000000,50.000000,393.000000
75%,4.046274e+06,0.000000,0.990000,7.000000,11.000000,55.000000,64.000000,1658.000000
max,6.114330e+06,16.000000,2.010000,15.000000,100.000000,100.000000,230.000000,27000.000000


IDpol         0
ClaimNb       0
Exposure      0
Area          0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Density       0
Region        0
dtype: int64

# 4. Define Target Variable and Predictor Variables

The objective of this notebook is to predict **claim frequency**, defined as the number of claims reported by each policyholder.

The target variable is separated from the explanatory variables to prepare the data for preprocessing and model training.

The exposure variable is retained separately because it represents the insured period and will later be incorporated into the pricing framework.

In [4]:
# Target variable
y = df["ClaimNb"]

# Exposure
exposure = df["Exposure"]

# Features
X = df.drop(columns=["ClaimNb"])

# 5. Identify Feature Types

Machine learning preprocessing depends on the type of each variable.

The predictor variables are divided into:

- Numerical variables
- Categorical variables

This separation allows us to apply different preprocessing techniques to each group.

In [5]:
categorical_features = X.select_dtypes(include=["object", "category"]).columns

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

print("Categorical Features")
print(list(categorical_features))

print("\nNumerical Features")
print(list(numerical_features))

Categorical Features
['Area', 'VehBrand', 'VehGas', 'Region']

Numerical Features
['IDpol', 'Exposure', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'Density']


# 6. Data Preprocessing Pipeline

Real-world insurance datasets frequently contain:

- Missing numerical values
- Missing categorical values
- Categorical rating factors

To prepare the dataset for machine learning, the following preprocessing steps are applied:

### Numerical Variables

- Median imputation

### Categorical Variables

- Most frequent category imputation
- One-Hot Encoding

A preprocessing pipeline ensures that identical transformations are applied to both the training and testing datasets.

In [8]:
# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# 7. Train-Test Split

To evaluate the generalization performance of the model, the dataset is divided into training and testing subsets.

The data is split as follows:

- **Training Set:** 80%
- **Testing Set:** 20%

The model is trained exclusively on the training data, while the testing data remains unseen during training. This approach provides an unbiased estimate of predictive performance on new policyholders.

A fixed random seed is used to ensure that the results are reproducible.

In [9]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test, exposure_train, exposure_test = train_test_split(
    X,
    y,
    exposure,
    test_size=0.20,
    random_state=42
)

print("Training observations :", X_train.shape[0])
print("Testing observations  :", X_test.shape[0])
print("Training features     :", X_train.shape[1])

Training observations : 542410
Testing observations  : 135603
Training features     : 11


# 8. Apply Data Preprocessing

The preprocessing pipeline developed earlier is now fitted on the training data and subsequently applied to both training and testing datasets.

This process includes:

- Median imputation for numerical variables
- Most frequent imputation for categorical variables
- One-Hot Encoding of categorical variables

The preprocessing transformations are learned **only from the training dataset** to prevent data leakage.

In [10]:
# Fit preprocessing pipeline on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing data
X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape :", X_train_processed.shape)
print("Processed Testing Shape  :", X_test_processed.shape)

Processed Training Shape : (542410, 48)
Processed Testing Shape  : (135603, 48)


# 9. Configure the XGBoost Frequency Model

An XGBoost regression model is used to predict claim frequency.

Unlike traditional Generalized Linear Models (GLMs), XGBoost can automatically capture:

- Nonlinear relationships
- Complex feature interactions
- Threshold effects
- High-order dependencies among predictor variables

Since claim frequency represents count data, the **Poisson objective function** is selected.

The selected hyperparameters provide a balance between predictive performance and model complexity while reducing the risk of overfitting.

In [11]:
# Initialize XGBoost model
xgb_model = XGBRegressor(
    objective="count:poisson",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# 10. Train the XGBoost Model

The XGBoost model is trained using the processed training dataset.

During training, the gradient boosting algorithm sequentially builds decision trees, where each tree attempts to correct the prediction errors made by the previous trees.

This iterative learning process enables the model to capture complex patterns in insurance claim frequency.

In [12]:
# Train the model
xgb_model.fit(
    X_train_processed,
    y_train
)

print("Model training completed successfully.")

Model training completed successfully.


# 11. Generate Claim Frequency Predictions

After training, the fitted model is applied to the testing dataset to estimate the expected claim frequency for each policyholder.

These predictions represent the model's estimate of the average number of claims expected over the exposure period.

In [13]:
# Predict claim frequency
y_pred = xgb_model.predict(X_test_processed)

# Ensure predictions remain non-negative
y_pred = np.clip(y_pred, 0, None)

# Display sample predictions
prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

prediction_df.head()

,Actual,Predicted
0,0,0.049596
1,0,0.029851
2,0,0.050562
3,0,0.070150
4,0,0.025906


# 12. Model Performance Evaluation

To assess predictive performance, three evaluation metrics are calculated:

- **Root Mean Squared Error (RMSE):** Measures the average magnitude of prediction errors while giving greater weight to larger errors.
- **Mean Absolute Error (MAE):** Measures the average absolute prediction error and is less sensitive to extreme values.
- **Poisson Deviance:** A distribution-specific metric commonly used for evaluating count models such as insurance claim frequency.

These metrics enable direct comparison with the Poisson GLM and Negative Binomial GLM developed in previous stages of the project.

In [14]:
# Calculate evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
poisson_dev = mean_poisson_deviance(y_test, y_pred)

# Display results
performance = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "Poisson Deviance"],
    "Value": [rmse, mae, poisson_dev]
})

performance

,Metric,Value
0,RMSE,0.205964
1,MAE,0.074728
2,Poisson Deviance,0.235683


# 13. Interpretation of Model Performance

The predictive performance of the XGBoost frequency model was evaluated using three complementary metrics: Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and Poisson Deviance.

| Metric | Value |
|--------|-------:|
| RMSE | **0.205964** |
| MAE | **0.074728** |
| Poisson Deviance | **0.235683** |

## Interpretation

### Root Mean Squared Error (RMSE)

The model achieved an **RMSE of 0.206**, indicating that the average prediction error is relatively small. Since RMSE penalizes larger errors more heavily than MAE, this result suggests that the model produces stable predictions with few large deviations.

### Mean Absolute Error (MAE)

The **MAE of 0.0747** indicates that, on average, the predicted claim frequency differs from the observed claim frequency by approximately **0.075 claims per policy**. Given that motor insurance claim frequency is typically low, this represents a relatively accurate level of prediction.

### Poisson Deviance

The **Poisson Deviance of 0.2357** measures model performance under the Poisson distribution, which is appropriate for count data such as insurance claims. Lower deviance values indicate a better fit between predicted and observed claim counts. The obtained value suggests that the model captures the underlying claim frequency reasonably well.

## Overall Assessment

The evaluation metrics indicate that the XGBoost model provides accurate predictions of insurance claim frequency while effectively modeling the count nature of the response variable.

Compared with traditional Generalized Linear Models, XGBoost is expected to better capture nonlinear relationships and complex interactions among policyholder and vehicle characteristics. A direct comparison with the Poisson and Negative Binomial GLMs will determine whether these improvements translate into superior predictive performance.

In [15]:
performance = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "Poisson Deviance"],
    "Value": [rmse, mae, poisson_dev]
})

performance["Value"] = performance["Value"].round(4)

performance

,Metric,Value
0,RMSE,0.2060
1,MAE,0.0747
2,Poisson Deviance,0.2357


In [21]:
id="xgb_check"
%whos

Variable                  Type                 Data/Info
--------------------------------------------------------
ColumnTransformer         ABCMeta              <class 'sklearn.compose._<...>ormer.ColumnTransformer'>
OneHotEncoder             type                 <class 'sklearn.preproces<...>_encoders.OneHotEncoder'>
Pipeline                  ABCMeta              <class 'sklearn.pipeline.Pipeline'>
SimpleImputer             type                 <class 'sklearn.impute._base.SimpleImputer'>
X                         DataFrame            Shape: (678013, 11)
XGBRegressor              type                 <class 'xgboost.sklearn.XGBRegressor'>
X_test                    DataFrame            Shape: (135603, 11)
X_test_processed          csr_matrix           <Compressed Sparse Row sp<...>)	1.0\n  (135602, 26)	1.0
X_train                   DataFrame            Shape: (542410, 11)
X_train_processed         csr_matrix           <Compressed Sparse Row sp<...>)	1.0\n  (542409, 43)	1.0
categorical_

In [22]:
import pandas as pd

xgb_frequency_output = pd.DataFrame({
    "Actual_Frequency": y_test.values,
    "XGB_Frequency": y_pred
})


xgb_frequency_output.to_csv(
    "xgb_frequency_predictions.csv",
    index=False
)


print("XGBoost frequency predictions exported successfully.")

XGBoost frequency predictions exported successfully.


In [23]:
type(xgb_model)

xgboost.sklearn.XGBRegressor

In [24]:
type(X_test_processed)

scipy.sparse._csr.csr_matrix

In [25]:
import shap

explainer = shap.TreeExplainer(xgb_model)

shap_values = explainer.shap_values(X_test_processed)

In [27]:
%whos

Variable                  Type                 Data/Info
--------------------------------------------------------
ColumnTransformer         ABCMeta              <class 'sklearn.compose._<...>ormer.ColumnTransformer'>
OneHotEncoder             type                 <class 'sklearn.preproces<...>_encoders.OneHotEncoder'>
Pipeline                  ABCMeta              <class 'sklearn.pipeline.Pipeline'>
SimpleImputer             type                 <class 'sklearn.impute._base.SimpleImputer'>
X                         DataFrame            Shape: (678013, 11)
XGBRegressor              type                 <class 'xgboost.sklearn.XGBRegressor'>
X_test                    DataFrame            Shape: (135603, 11)
X_test_processed          csr_matrix           <Compressed Sparse Row sp<...>)	1.0\n  (135602, 26)	1.0
X_train                   DataFrame            Shape: (542410, 11)
X_train_processed         csr_matrix           <Compressed Sparse Row sp<...>)	1.0\n  (542409, 43)	1.0
categorical_

In [28]:
feature_names = preprocessor.get_feature_names_out()

len(feature_names)

48

In [29]:
feature_names[:10]

array(['num__IDpol', 'num__Exposure', 'num__VehPower', 'num__VehAge',
       'num__DrivAge', 'num__BonusMalus', 'num__Density', 'cat__Area_A',
       'cat__Area_B', 'cat__Area_C'], dtype=object)

In [30]:
import joblib

joblib.dump(
    xgb_model,
    "xgb_frequency_model.pkl"
)

print("XGBoost model saved successfully")

XGBoost model saved successfully


In [31]:
joblib.dump(
    preprocessor,
    "xgb_preprocessor.pkl"
)

['xgb_preprocessor.pkl']